In [4]:
# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026 v3)
# ============================================================

# 1. CARGA DE LIBRERÍAS
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales", "effsize", "ggtext")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN DE RUTAS
# ===============================
paths <- list(
  metabolic    = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_TumorPhenotype_PCA_metrics_actualizado_sinl2/Merged_TumorPhenotype_PCA_AllData.csv",
  clinicaldata = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_PCA_seleccion_variables/DATA_MASTER_CLUSTERS_Y_CLINICA.csv",
  div          = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_divergentes_para_estudio_pyn.csv",
  core         = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_core_correlacion_Paretoynormas.csv"
)

group_colors <- c("Core" = "#0072B2", "Divergent" = "#D55E00")
COL_DIV  <- "#D55E00"   # Divergent > Core
COL_CORE <- "#0072B2"   # Core > Divergent

# ─── TEMA GLOBAL ─────────────────────────────────────────────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 23, face = "bold"),
    axis.title    = element_text(size = 20),
    axis.text     = element_text(size = 18),
    legend.text   = element_text(size = 18),
    legend.title  = element_text(size = 19, face = "bold"),
    strip.text    = element_text(size = 19, face = "bold"),
    plot.subtitle = element_text(size = 18, color = "grey50")
  )

# ===============================
# 3. VARIABLES METABÓLICAS
# ===============================
SOLS <- c("FBA", "pFBA", "L1w", "L2", "L2w")

expand_sols <- function(roots, sols = SOLS) {
  as.vector(outer(roots, sols, paste, sep = "_"))
}

SCALAR_ROOTS <- c(
  "CU", "EA", "WarburgIndex",
  "ATPConsumption", "ATPProduction",
  "RedoxIndex", "MFI", "AnabolismScore",
  "NADPHdemand", "TCA_completeness",
  "LipidSat", "LipidUnsat", "LipidPL",
  "GlnDependence"
)
SCALAR_VARS <- expand_sols(SCALAR_ROOTS)

ONCOMET_NAMES <- c("Lactate", "Succinate", "AlphaKG")
ONCOMET_VARS  <- as.vector(outer(
  paste0("Oncomet_", ONCOMET_NAMES), SOLS, paste, sep = "_"
))

ALL_METAB_VARS_STATIC <- c(SCALAR_VARS, ONCOMET_VARS)

# ===============================
# 4. PREPROCESAMIENTO Y CRUCE
# ===============================
clean_names <- function(x) toupper(gsub("\\.", "-", trimws(as.character(x))))

df_metab <- read.csv(paths$metabolic,    stringsAsFactors = FALSE)
df_clin  <- read.csv(paths$clinicaldata, stringsAsFactors = FALSE)
df_div   <- read.csv(paths$div,          stringsAsFactors = FALSE)
df_core  <- read.csv(paths$core,         stringsAsFactors = FALSE)

df_metab$ModelName <- clean_names(df_metab$ModelName)
df_clin$ModelName  <- clean_names(df_clin$ModelName)
div_names          <- clean_names(df_div$ModelName)
core_names         <- clean_names(df_core$ModelName)

sa_cols_csv <- grep(
  paste0("^SA_.+_(", paste(SOLS, collapse = "|"), ")$"),
  colnames(df_metab), value = TRUE
)
cat("\n🔍 Subsistemas SA detectados en CSV:", length(sa_cols_csv), "\n")
if (length(sa_cols_csv) > 0) {
  subsys_unique <- unique(sub(paste0("_(", paste(SOLS, collapse="|"), ")$"), "", sa_cols_csv))
  cat("   Subsistemas únicos:", length(subsys_unique), "\n")
  cat(paste0("   • ", head(subsys_unique, 10), collapse = "\n"), "\n")
  if (length(subsys_unique) > 10) cat("   ...", length(subsys_unique) - 10, "más\n")
}

ALL_METAB_VARS <- c(ALL_METAB_VARS_STATIC, sa_cols_csv)

overlap <- intersect(div_names, core_names)
if (length(overlap) > 0) {
  warning(paste0("⚠️  ", length(overlap), " pacientes en ambos grupos (se excluirán): ",
                 paste(head(overlap, 5), collapse = ", ")))
}

analysis <- df_metab %>%
  mutate(Group = case_when(
    ModelName %in% overlap    ~ NA_character_,
    ModelName %in% div_names  ~ "Divergent",
    ModelName %in% core_names ~ "Core",
    TRUE                      ~ "Other"
  )) %>%
  filter(Group %in% c("Core", "Divergent"))

metab_available <- intersect(ALL_METAB_VARS, colnames(analysis))
metab_missing   <- setdiff(ALL_METAB_VARS,  colnames(analysis))

cat("\n📊 Variables metabólicas encontradas:", length(metab_available), "/", length(ALL_METAB_VARS))
if (length(metab_missing) > 0) {
  cat("\n⚠️  Variables NO encontradas (", length(metab_missing), "):\n")
  cat(paste0("   • ", head(metab_missing, 20), collapse = "\n"), "\n")
  if (length(metab_missing) > 20) cat("   ...", length(metab_missing) - 20, "más\n")
}

# ─── Join clínico ────────────────────────────────────────────────────────────
race_col_clin <- grep("^race",        colnames(df_clin), ignore.case = TRUE, value = TRUE)[1]
sex_col_clin  <- grep("^sex|^gender", colnames(df_clin), ignore.case = TRUE, value = TRUE)[1]

clinical_cols <- unique(c(
  "ModelName",
  "ajcc_pathologic_stage.diagnoses", "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",     "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",           "morphology.diagnoses",
  "primary_diagnosis.diagnoses",     "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses", "prior_treatment.diagnoses",
  "sample_type.samples",             "tissue_type.samples",
  "tumor_descriptor.samples",        "specimen_type.samples",
  "is_ffpe.samples",                 "oct_embedded.samples",
  "age_at_diagnosis.diagnoses",      "alcohol_history.exposures",
  "OS.time", "OS",
  "Menopausal.Status", "Cancer.Type",
  "ER", "PR", "HER2", "Subtype", "Genetic.Ancestry",
  "Prior_Treatment_Flag", "Metastasis_Flag",
  "ER_PR_HER2_Combo", "ER_PR_HER2_Combo_encoded", "Subtype_encoded",
  race_col_clin, sex_col_clin
)[!is.na(c(
  "ModelName",
  "ajcc_pathologic_stage.diagnoses", "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",     "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",           "morphology.diagnoses",
  "primary_diagnosis.diagnoses",     "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses", "prior_treatment.diagnoses",
  "sample_type.samples",             "tissue_type.samples",
  "tumor_descriptor.samples",        "specimen_type.samples",
  "is_ffpe.samples",                 "oct_embedded.samples",
  "age_at_diagnosis.diagnoses",      "alcohol_history.exposures",
  "OS.time", "OS",
  "Menopausal.Status", "Cancer.Type",
  "ER", "PR", "HER2", "Subtype", "Genetic.Ancestry",
  "Prior_Treatment_Flag", "Metastasis_Flag",
  "ER_PR_HER2_Combo", "ER_PR_HER2_Combo_encoded", "Subtype_encoded",
  race_col_clin, sex_col_clin
))])

clin_subset    <- df_clin %>% select(any_of(clinical_cols))
cols_from_clin <- setdiff(colnames(clin_subset), "ModelName")

analysis <- analysis %>%
  select(-any_of(cols_from_clin)) %>%
  left_join(clin_subset, by = "ModelName") %>%
  rename_with(~ gsub("Menopausal\\.Status", "Menopausal Status", .x)) %>%
  rename_with(~ gsub("Cancer\\.Type",       "Cancer Type",       .x)) %>%
  rename_with(~ gsub("Genetic\\.Ancestry",  "Genetic Ancestry",  .x))

if (!is.na(race_col_clin) && race_col_clin %in% colnames(analysis) && race_col_clin != "Race")
  analysis <- analysis %>% rename(Race = !!sym(race_col_clin))
if (!is.na(sex_col_clin)  && sex_col_clin  %in% colnames(analysis) && sex_col_clin  != "Sex")
  analysis <- analysis %>% rename(Sex  = !!sym(sex_col_clin))

# ── Verificación de variables clave ──────────────────────────────────────────
key_vars <- c("OS.time", "OS", "Subtype", "ER", "PR", "HER2",
              "Genetic Ancestry", "Sex", "Race", "Menopausal Status")
cat("\n📋 Verificación de variables clave en analysis:\n")
for (v in key_vars) {
  if (v %in% colnames(analysis)) {
    n_ok <- sum(!is.na(analysis[[v]]))
    cat(sprintf("   ✅ %-30s → %d/%d con datos (%.1f%%)\n",
                v, n_ok, nrow(analysis), n_ok / nrow(analysis) * 100))
  } else {
    cat(sprintf("   ❌ %-30s → NO encontrada\n", v))
  }
}

cat("\n✅ Pacientes Core:", sum(analysis$Group == "Core"),
    " | Divergent:", sum(analysis$Group == "Divergent"), "\n")

# ===============================
# 5A. GRUPOS CLÍNICOS
# ===============================
CLIN_GROUP_A <- intersect(c(
  "ER","PR","HER2","Subtype","ER_PR_HER2_Combo",
  "tumor_grade.diagnoses","morphology.diagnoses","primary_diagnosis.diagnoses"
), colnames(analysis))

CLIN_GROUP_B <- intersect(c(
  "ajcc_pathologic_stage.diagnoses","ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses","ajcc_pathologic_m.diagnoses","Metastasis_Flag"
), colnames(analysis))

CLIN_GROUP_C <- intersect(c(
  "prior_treatment.diagnoses","treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses","Prior_Treatment_Flag"
), colnames(analysis))

CLIN_GROUP_D <- intersect(c(
  "age_at_diagnosis.diagnoses","Menopausal Status","Genetic Ancestry",
  "alcohol_history.exposures","Race","Sex","Cancer Type",
  "OS.time", "OS"
), colnames(analysis))

CLIN_GROUPS <- Filter(function(v) length(v) > 0, list(
  "🧬 A. Biología Tumoral"        = CLIN_GROUP_A,
  "📊 B. Estadio y Carga Tumoral" = CLIN_GROUP_B,
  "💊 C. Historia Terapéutica"    = CLIN_GROUP_C,
  "🧍 D. Factores del Paciente"   = CLIN_GROUP_D
))

cat("\n📂 Grupos clínicos activos:", length(CLIN_GROUPS), "\n")
for (g in names(CLIN_GROUPS))
  cat("   •", g, ":", length(CLIN_GROUPS[[g]]), "variables\n")

# ===============================
# 5B. GRUPOS METABÓLICOS
# ===============================
make_group <- function(roots, available, sols = SOLS)
  intersect(expand_sols(roots, sols), available)

METAB_GROUPS <- Filter(function(v) length(v) > 0, list(
  "🔥 1. Energía y Bioenergética" = make_group(
    c("ATPConsumption","ATPProduction","WarburgIndex","MFI","CU","EA"), metab_available),
  "🔴 2. Estado Redox y Anabolismo" = make_group(
    c("RedoxIndex","NADPHdemand","AnabolismScore","TCA_completeness"), metab_available),
  "🧈 3. Metabolismo Lipídico" = make_group(
    c("LipidSat","LipidUnsat","LipidPL"), metab_available),
  "🧪 4. Dependencias Metabólicas Específicas" = c(
    make_group("GlnDependence", metab_available),
    intersect(ONCOMET_VARS, metab_available)),
  "🧩 5. Reprogramación Sistémica (SA_*)" = sa_cols_csv[sa_cols_csv %in% metab_available]
))

cat("\n📂 Grupos metabólicos activos:", length(METAB_GROUPS), "\n")
for (g in names(METAB_GROUPS))
  cat("   •", g, ":", length(METAB_GROUPS[[g]]), "variables\n")

# ===============================
# 6. ANÁLISIS ESTADÍSTICO (FDR) + CLIFF'S DELTA
# ===============================

# Cliff's delta: positivo = Divergent > Core
calc_cliffs_delta <- function(x_div, x_core) {
  nx <- length(x_div); ny <- length(x_core)
  if (nx == 0 || ny == 0) return(NA_real_)
  sum(outer(x_div, x_core, FUN = function(a, b) sign(a - b))) / (nx * ny)
}

label_effect <- function(d) {
  case_when(
    abs(d) >= 0.474 ~ "Grande",
    abs(d) >= 0.330 ~ "Mediano",
    abs(d) >= 0.147 ~ "Pequeño",
    TRUE            ~ "Negligible"
  )
}

fmt_stars <- function(p) {
  case_when(
    p < 0.001 ~ "***",
    p < 0.01  ~ "**",
    p < 0.05  ~ "*",
    TRUE      ~ "ns"
  )
}

run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))
    group_sizes <- table(sub_data$Group)
    if (length(group_sizes) < 2 || any(group_sizes < 3)) return(NULL)

    test     <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)
    meds     <- sub_data %>% group_by(Group) %>%
      summarise(m = median(value), .groups = "drop")
    med_core <- meds$m[meds$Group == "Core"]
    med_div  <- meds$m[meds$Group == "Divergent"]
    x_div    <- sub_data$value[sub_data$Group == "Divergent"]
    x_core   <- sub_data$value[sub_data$Group == "Core"]
    delta    <- calc_cliffs_delta(x_div, x_core)
    pct_diff <- ifelse(med_core == 0, NA_real_,
                       (med_div - med_core) / abs(med_core) * 100)

    data.frame(
      Feature      = col,
      Med_Core     = med_core,
      Med_Div      = med_div,
      p_val        = test$p.value,
      cliffs_delta = delta,
      Pct_Diff     = pct_diff
    )
  })
  if (nrow(results) == 0) return(results)
  results %>%
    mutate(
      p_adj       = p.adjust(p_val, method = "fdr"),
      Fold_Change = Med_Div / ifelse(Med_Core == 0, NA, Med_Core),
      sig_stars   = fmt_stars(p_adj),
      effect_size = label_effect(cliffs_delta)
    ) %>%
    arrange(p_adj)
}

metab_res <- run_stats(analysis, metab_available)
cat("\n📈 Variables metabólicas significativas (p_adj < 0.05):",
    sum(metab_res$p_adj < 0.05, na.rm = TRUE), "/", nrow(metab_res), "\n")
write.csv(metab_res, "Estadisticas_Metabolicas_CoreVsDivergent.csv", row.names = FALSE)

summary_by_group <- map_df(names(METAB_GROUPS), function(g) {
  vars_g <- METAB_GROUPS[[g]]
  res_g  <- metab_res %>% filter(Feature %in% vars_g)
  if (nrow(res_g) == 0) return(NULL)
  data.frame(
    Grupo         = g,
    n_vars        = length(vars_g),
    n_sig_p05     = sum(res_g$p_adj < 0.05, na.rm = TRUE),
    n_sig_p01     = sum(res_g$p_adj < 0.01, na.rm = TRUE),
    mejor_feature = res_g$Feature[which.min(res_g$p_adj)],
    mejor_p_adj   = min(res_g$p_adj, na.rm = TRUE),
    max_abs_delta = max(abs(res_g$cliffs_delta), na.rm = TRUE)
  )
})
write.csv(summary_by_group, "Resumen_Significancia_PorGrupo.csv", row.names = FALSE)
print(summary_by_group)

# Variables clínicas numéricas incluyendo OS.time y OS
numeric_clinical_cols <- intersect(
  c("age_at_diagnosis.diagnoses", "is_ffpe.samples", "oct_embedded.samples",
    "Prior_Treatment_Flag", "Metastasis_Flag",
    "ER_PR_HER2_Combo_encoded", "Subtype_encoded",
    "OS.time", "OS"),
  colnames(analysis)
)
clinical_num_res <- run_stats(analysis, numeric_clinical_cols)

cat("\n📈 Variables clínicas numéricas analizadas:", length(numeric_clinical_cols), "\n")
if (nrow(clinical_num_res) > 0) {
  cat("   Significativas (p_adj < 0.05):",
      sum(clinical_num_res$p_adj < 0.05, na.rm = TRUE), "\n")
  print(clinical_num_res %>% select(Feature, Med_Core, Med_Div,
                                     cliffs_delta, p_adj, sig_stars, effect_size))
}

# ===============================
# 7. FUNCIONES DE PLOTEO
# ===============================

# ── 7A. Violin + Boxplot individual (subtítulo con δ) ───────────────────────
plot_box_feat <- function(feat, data, stats_df) {
  if (!feat %in% colnames(data)) return(NULL)
  p_info <- stats_df %>% filter(Feature == feat) %>% slice(1)
  if (nrow(p_info) == 0) return(NULL)

  sig_color    <- if (!is.na(p_info$p_adj) && p_info$p_adj < 0.05) "#e63946" else "grey60"
  subtitle_txt <- paste0(
    "p-adj: ", signif(p_info$p_adj, 3),
    "  |  δ=", round(p_info$cliffs_delta, 2),
    " (", p_info$effect_size, ")",
    if (!is.na(p_info$Pct_Diff)) paste0("  |  Δ%: ", round(p_info$Pct_Diff, 1)) else ""
  )

  # ── Detectar variable binaria (0/1) ──────────────────────────────────────
  plot_data <- data %>% filter(!is.na(Group))
  vals <- sort(unique(na.omit(plot_data[[feat]])))
  is_binary <- is.numeric(plot_data[[feat]]) &&
               length(vals) == 2 && all(vals %in% c(0, 1))

  # ── Detectar variable continua con missings relevantes ───────────────────
  is_survival_time <- grepl("OS\\.time|survival.*time|time.*survival",
                             feat, ignore.case = TRUE)
  is_survival_status <- feat == "OS" ||
                        grepl("survival.*status|status.*survival",
                              feat, ignore.case = TRUE)
  is_sex <- grepl("^sex$|^gender$", feat, ignore.case = TRUE)

  # ══════════════════════════════════════════════════════════════════════════
  # CASO 1: OS / Survival Status → binaria + barra Missing
  # ══════════════════════════════════════════════════════════════════════════
  if (is_survival_status || is_binary) {

    plot_data <- plot_data %>%
      mutate(Category = case_when(
        is.na(!!sym(feat))               ~ "Missing",
        is_survival_status & !!sym(feat) == 1 ~ "Fallecido (1)",
        is_survival_status & !!sym(feat) == 0 ~ "Vivo (0)",
        !!sym(feat) == 1                 ~ "Sí (1)",
        !!sym(feat) == 0                 ~ "No (0)",
        TRUE                             ~ "Missing"
      )) %>%
      mutate(Category = factor(Category,
               levels = c(if (is_survival_status) c("Vivo (0)", "Fallecido (1)")
                          else c("No (0)", "Sí (1)"), "Missing")))

    tbl   <- table(plot_data$Category[plot_data$Category != "Missing"],
                   plot_data$Group[plot_data$Category != "Missing"])
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    chi_label <- if (!is.na(p_chi)) paste0("Chi² p=", signif(p_chi, 3)) else ""

    count_df <- plot_data %>%
      group_by(Group, Category) %>%
      summarise(n = n(), .groups = "drop") %>%
      group_by(Group) %>%
      mutate(prop  = n / sum(n),
             label = paste0(n, "\n(", round(prop * 100, 1), "%)"))

    fill_vals  <- if (is_survival_status) c("Vivo (0)"     = COL_CORE,
                                             "Fallecido (1)"= COL_DIV,
                                             "Missing"      = "grey80")
                  else                    c("No (0)"        = COL_CORE,
                                             "Sí (1)"       = COL_DIV,
                                             "Missing"      = "grey80")

    ggplot(count_df, aes(x = Group, y = prop, fill = Category)) +
      geom_col(position = "stack", color = "white", linewidth = 0.4) +
      geom_text(aes(label = label),
                position = position_stack(vjust = 0.5),
                size = 4.2, fontface = "bold", color = "white") +
      scale_fill_manual(values = fill_vals, name = NULL) +
      scale_y_continuous(labels = percent) +
      labs(title    = feat,
           subtitle = paste0(subtitle_txt, "  |  ", chi_label),
           x = "Grupo", y = "Proporción") +
      base_theme +
      theme(legend.position = "right",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 12))

  # ══════════════════════════════════════════════════════════════════════════
  # CASO 2: Sex / Gender → categórica + barra Missing
  # ══════════════════════════════════════════════════════════════════════════
  } else if (is_sex) {

    plot_data <- plot_data %>%
      mutate(Category = case_when(
        is.na(!!sym(feat)) | trimws(tolower(!!sym(feat))) %in% c("","na","nan","unknown") ~ "Missing",
        TRUE ~ stringr::str_to_title(trimws(as.character(!!sym(feat))))
      ))

    count_df <- plot_data %>%
      group_by(Group, Category) %>%
      summarise(n = n(), .groups = "drop") %>%
      group_by(Group) %>%
      mutate(prop  = n / sum(n),
             label = paste0(n, "\n(", round(prop * 100, 1), "%)"))

    n_cats  <- length(unique(count_df$Category))
    pal     <- c(setNames(scales::hue_pal()(n_cats - 1),
                          setdiff(unique(count_df$Category), "Missing")),
                 "Missing" = "grey80")

    tbl   <- table(plot_data$Category[plot_data$Category != "Missing"],
                   plot_data$Group[plot_data$Category != "Missing"])
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    chi_label <- if (!is.na(p_chi)) paste0("Chi² p=", signif(p_chi, 3)) else ""

    ggplot(count_df, aes(x = Group, y = prop, fill = Category)) +
      geom_col(position = "stack", color = "white", linewidth = 0.4) +
      geom_text(aes(label = label),
                position = position_stack(vjust = 0.5),
                size = 4.2, fontface = "bold", color = "white") +
      scale_fill_manual(values = pal, name = feat) +
      scale_y_continuous(labels = percent) +
      labs(title    = feat,
           subtitle = paste0(subtitle_txt, "  |  ", chi_label),
           x = "Grupo", y = "Proporción") +
      base_theme +
      theme(legend.position = "right",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 12))

  # ══════════════════════════════════════════════════════════════════════════
  # CASO 3: OS.time / Survival Time → continua con panel Missing separado
  # ══════════════════════════════════════════════════════════════════════════
  } else if (is_survival_time) {

    plot_data <- plot_data %>%
      mutate(
        has_data = !is.na(!!sym(feat)) & is.finite(!!sym(feat)),
        value    = as.numeric(!!sym(feat))
      )

    n_miss <- plot_data %>%
      group_by(Group) %>%
      summarise(n_miss = sum(!has_data), n_total = n(), .groups = "drop") %>%
      mutate(pct = round(n_miss / n_total * 100, 1),
             label = paste0(n_miss, "\n(", pct, "%)"))

    p1 <- ggplot(plot_data %>% filter(has_data),
                 aes(x = Group, y = value, fill = Group)) +
      geom_violin(alpha = 0.2, color = NA) +
      geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
      scale_fill_manual(values = group_colors) +
      labs(title = feat, subtitle = subtitle_txt,
           x = "Grupo", y = "Tiempo (días)") +
      base_theme +
      theme(legend.position = "none",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 12))

    p2 <- ggplot(n_miss, aes(x = Group, y = n_miss, fill = Group)) +
      geom_col(width = 0.5, alpha = 0.7) +
      geom_text(aes(label = label), vjust = -0.3,
                size = 4.5, fontface = "bold") +
      scale_fill_manual(values = group_colors) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +
      labs(title = "Missing", x = "Grupo", y = "N pacientes") +
      base_theme +
      theme(legend.position = "none",
            plot.title = element_text(size = 14, color = "grey50"))

    return(gridExtra::arrangeGrob(p1, p2, ncol = 2, widths = c(3, 1)))

  # ══════════════════════════════════════════════════════════════════════════
  # CASO 4: Continua estándar → violin + boxplot
  # ══════════════════════════════════════════════════════════════════════════
  } else {

    plot_data <- plot_data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
    ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
      geom_violin(alpha = 0.2, color = NA) +
      geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
      labs(title = feat, subtitle = subtitle_txt, y = NULL) +
      scale_fill_manual(values = group_colors) +
      base_theme +
      theme(legend.position = "none",
            plot.title    = element_text(size = 17, color = sig_color, face = "bold"),
            plot.subtitle = element_text(size = 13))
  }
}

# ── 7C. Forest plot de Cliff's delta ────────────────────────────────────────
plot_forest_cliffs <- function(stats_df, title_txt = "Forest Plot — Cliff's δ",
                               top_n = 40) {
  if (is.null(stats_df) || nrow(stats_df) == 0) return(NULL)

  df <- stats_df %>%
    filter(!is.na(cliffs_delta), !is.na(p_adj)) %>%
    arrange(p_adj) %>%
    head(top_n) %>%
    mutate(
      pt_color  = case_when(
        p_adj >= 0.05        ~ "grey65",
        cliffs_delta > 0     ~ COL_DIV,
        TRUE                 ~ COL_CORE
      ),
      ann_label = paste0(fmt_stars(p_adj),
                         "  δ=", round(cliffs_delta, 2),
                         "  (", round(Pct_Diff, 1), "%)"),
      Feature   = factor(Feature, levels = Feature[order(cliffs_delta)])
    )

  ggplot(df, aes(x = cliffs_delta, y = Feature)) +
    annotate("rect", xmin = -0.474, xmax =  0.474,
             ymin = -Inf, ymax = Inf, fill = "grey96") +
    annotate("rect", xmin = -0.330, xmax =  0.330,
             ymin = -Inf, ymax = Inf, fill = "grey92") +
    annotate("rect", xmin = -0.147, xmax =  0.147,
             ymin = -Inf, ymax = Inf, fill = "grey87") +
    geom_vline(xintercept = 0, linetype = "solid",
               color = "grey40", linewidth = 0.6) +
    geom_segment(aes(x = 0, xend = cliffs_delta,
                     yend = Feature, color = pt_color),
                 linewidth = 0.7, alpha = 0.6) +
    geom_point(aes(color = pt_color, size = abs(cliffs_delta)),
               alpha = 0.95) +
    geom_text(aes(label = ann_label, color = pt_color,
                  hjust = ifelse(cliffs_delta >= 0, -0.12, 1.12)),
              size = 5.5, fontface = "bold") +
    scale_color_identity() +
    scale_size_continuous(range = c(2.5, 8), guide = "none") +
    scale_x_continuous(
      limits = c(-1.1, 1.1),
      breaks = c(-0.6, -0.474, -0.330, -0.147, 0, 0.147, 0.330, 0.474, 0.6),
      labels = c("-0.6", "Grande", "Mediano", "Pequeño",
                 "0", "Pequeño", "Mediano", "Grande", "0.6")
    ) +
    labs(
      title    = title_txt,
      subtitle = "Cliff's δ + Δ% mediana | <span style='color:#0072B2'>●</span> mayor en Core · <span style='color:#D55E00'>●</span> mayor en Divergent",
      x        = "Cliff's delta  [Core vs Divergent]",
      y        = NULL
    ) +
    theme_classic(base_size = 16) +
    theme(
      plot.title         = element_text(size = 18, face = "bold", hjust = 0.5),
      plot.subtitle      = element_markdown(size = 14, hjust = 0.5, color = "grey45"),
      axis.text.y        = element_text(size = 14),
      axis.text.x        = element_text(size = 11, angle = 35, hjust = 1),
      panel.grid.major.x = element_line(color = "grey88", linewidth = 0.4)
    )
}

# ── 7D. Clínica categórica ───────────────────────────────────────────────────
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)
  tmp[[var]] <- trimws(as.character(tmp[[var]]))
  if (is.numeric(data[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 6) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, x = "") +
      base_theme
  } else {
    tbl   <- table(tmp[[var]], tmp$Group)
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    subtitle_txt <- if (!is.na(p_chi)) paste("Chi² p:", signif(p_chi, 3)) else ""
    ggplot(tmp, aes(x = !!sym(var), fill = Group)) +
      geom_bar(position = "fill") +
      scale_y_continuous(labels = percent) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, subtitle = subtitle_txt, y = "Proporción") +
      base_theme +
      theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 15))
  }
}

# ── 7E. Integración metabólica × clínica ────────────────────────────────────
plot_metab_vs_clinical <- function(metab_feat, clin_var, data) {
  if (!metab_feat %in% colnames(data)) return(NULL)
  if (!clin_var   %in% colnames(data)) return(NULL)
  tmp <- data %>%
    filter(!is.na(!!sym(metab_feat)), !is.na(!!sym(clin_var)),
           is.finite(!!sym(metab_feat))) %>%
    mutate(ClinVar = trimws(as.character(!!sym(clin_var))))
  if (nrow(tmp) == 0 || length(unique(tmp$ClinVar)) < 2) return(NULL)
  ggplot(tmp, aes(x = ClinVar, y = !!sym(metab_feat), fill = Group)) +
    geom_boxplot(outlier.shape = 16, outlier.size = 1.5, alpha = 0.8) +
    scale_fill_manual(values = group_colors) +
    labs(title    = paste0(metab_feat, " × ", clin_var),
         subtitle = "¿La diferencia Core/Divergent es independiente del fenotipo clínico?",
         x = clin_var, y = metab_feat) +
    base_theme +
    theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 13))
}

# ===============================
# 8. GENERACIÓN DEL REPORTE PDF
# ===============================
pdf("Reporte_Metabolismo_Clinico_2026.pdf", width = 14, height = 11)

# ─── PÁGINA DE TÍTULO ────────────────────────────────────────────────────────
grid.newpage()
grid.text("Reporte: Core vs Divergent\nMetabolismo + Clínica + Supervivencia",
          gp = gpar(fontsize = 31, fontface = "bold"), y = 0.72)
grid.text(paste("Pacientes Core:", sum(analysis$Group == "Core"),
                "  |  Divergentes:", sum(analysis$Group == "Divergent")),
          gp = gpar(fontsize = 21), y = 0.58)
grid.text(paste("Variables metabólicas analizadas:", length(metab_available),
                "  |  Significativas (p_adj<0.05):", sum(metab_res$p_adj < 0.05, na.rm = TRUE)),
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.48)
grid.text(paste("Soluciones FBA:", paste(SOLS, collapse = ", ")),
          gp = gpar(fontsize = 17, col = "grey50"), y = 0.38)
grid.text("Estructura: Clínica → Metabolismo → Integración → Supervivencia",
          gp = gpar(fontsize = 17, col = "#0072B2"), y = 0.28)
grid.text("Tamaño de efecto: Cliff's δ  |  ≥0.147 pequeño · ≥0.330 mediano · ≥0.474 grande",
          gp = gpar(fontsize = 15, col = "grey55"), y = 0.18)

# ─── SECCIÓN I: CLÍNICA ──────────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN I\nDiferencias Clínicas: Core vs Divergent",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#0072B2"), y = 0.60)
grid.text("Macro-categorías: Biología Tumoral | Estadio | Historia Terapéutica | Factores del Paciente",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

for (clin_gname in names(CLIN_GROUPS)) {
  vars_clin    <- CLIN_GROUPS[[clin_gname]]
  numeric_vars <- vars_clin[sapply(vars_clin, function(v) is.numeric(analysis[[v]]))]
  categ_vars   <- setdiff(vars_clin, numeric_vars)

  # Numérico: violin+box con δ en subtítulo
  if (length(numeric_vars) > 0) {
    num_plots_clin <- map(numeric_vars, ~plot_box_feat(.x, analysis, clinical_num_res)) %>%
      Filter(Negate(is.null), .)
    if (length(num_plots_clin) > 0) {
      for (i in seq(1, length(num_plots_clin), by = 6)) {
        idx <- i:min(i+5, length(num_plots_clin))
        grid.arrange(grobs = num_plots_clin[idx],
                     ncol = min(3, length(idx)),
                     nrow = ceiling(length(idx)/min(3,length(idx))),
                     top  = textGrob(paste("Clínica Numérica —", clin_gname),
                                     gp = gpar(fontsize = 21, fontface = "bold")))
      }
    }

    clin_hm <- plot_heatmap_cliffs(
      clinical_num_res %>% filter(Feature %in% numeric_vars),
      group_name = paste("Clínica:", clin_gname),
      max_vars   = length(numeric_vars)
    )
    if (!is.null(clin_hm)) print(clin_hm)

    clin_forest <- plot_forest_cliffs(
      clinical_num_res %>% filter(Feature %in% numeric_vars),
      title_txt = paste("Cliff's δ — Clínica:", clin_gname),
      top_n     = length(numeric_vars)
    )
    if (!is.null(clin_forest)) print(clin_forest)
  }

  # Categórico: barras proporcionales
  if (length(categ_vars) > 0) {
    cat_plots_clin <- map(categ_vars, ~plot_clinical(.x, analysis)) %>%
      Filter(Negate(is.null), .)
    if (length(cat_plots_clin) > 0) {
      for (i in seq(1, length(cat_plots_clin), by = 2)) {
        idx <- i:min(i+1, length(cat_plots_clin))
        grid.arrange(grobs = cat_plots_clin[idx], nrow = length(idx), ncol = 1,
                     top = textGrob(paste("Clínica Categórica —", clin_gname),
                                    gp = gpar(fontsize = 21, fontface = "bold")))
      }
    }
  }
}

# ─── SECCIÓN II: METABOLISMO ─────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN II\nDiferencias Metabólicas: Core vs Divergent",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#D55E00"), y = 0.60)
grid.text("5 bloques funcionales: Energía | Redox | Lípidos | Dependencias | Reprogramación Sistémica",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

# Forest plot global Top 40
if (nrow(metab_res) > 0) {
  ff <- plot_forest_cliffs(metab_res, "Top 40 variables metabólicas — Cliff's δ", top_n = 40)
  if (!is.null(ff)) print(ff)
}

# Por grupo metabólico: heatmap δ + forest δ + violines
for (group_name in names(METAB_GROUPS)) {
  vars_in_group <- METAB_GROUPS[[group_name]]
  stats_group   <- metab_res %>% filter(Feature %in% vars_in_group)
  if (nrow(stats_group) == 0) next

  hm_g <- plot_heatmap_cliffs(stats_group, group_name, max_vars = 40)
  if (!is.null(hm_g)) print(hm_g)

  ff_g <- plot_forest_cliffs(stats_group,
                              title_txt = paste("Cliff's δ —", group_name),
                              top_n     = min(40, nrow(stats_group)))
  if (!is.null(ff_g)) print(ff_g)

  plots_g <- map(vars_in_group, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(plots_g) > 0) {
    for (i in seq(1, length(plots_g), by = 6)) {
      idx <- i:min(i+5, length(plots_g))
      grid.arrange(grobs = plots_g[idx],
                   ncol = min(3, length(idx)),
                   nrow = ceiling(length(idx)/min(3,length(idx))),
                   top  = textGrob(paste("Metabolismo —", group_name),
                                   gp = gpar(fontsize = 21, fontface = "bold")))
    }
  }
}

# Top 12 por p_adj + |δ|
top12 <- metab_res %>%
  filter(!is.na(p_adj), !is.na(cliffs_delta)) %>%
  arrange(p_adj, desc(abs(cliffs_delta))) %>%
  head(12)

if (nrow(top12) >= 2) {
  hm_top12 <- plot_heatmap_cliffs(top12, "Top 12 Variables", max_vars = 12)
  if (!is.null(hm_top12)) print(hm_top12)

  ff_top12 <- plot_forest_cliffs(top12,
                                  title_txt = "🏆 Top 12 Variables — Cliff's δ",
                                  top_n = 12)
  if (!is.null(ff_top12)) print(ff_top12)

  top12_plots <- map(top12$Feature, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(top12_plots) > 0) {
    for (i in seq(1, length(top12_plots), by = 6)) {
      idx <- i:min(i+5, length(top12_plots))
      grid.arrange(grobs = top12_plots[idx],
                   ncol = min(3, length(idx)),
                   nrow = ceiling(length(idx)/min(3,length(idx))),
                   top  = textGrob("🏆 Top 12 Variables Metabólicas (p_adj + Cliff's δ)",
                                   gp = gpar(fontsize = 23, fontface = "bold")))
    }
  }
}

# ─── SECCIÓN III: INTEGRACIÓN ────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN III\nAnálisis Integrado: Clínica ↔ Metabolismo",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#009E73"), y = 0.60)
grid.text("¿Core vs Divergent es reflejo del subtipo? ¿O reprogramación metabólica independiente?",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

integration_clin_vars <- intersect(
  c("Subtype","ajcc_pathologic_stage.diagnoses","ER","HER2","Metastasis_Flag"),
  colnames(analysis)
)
top_features <- top12$Feature

for (clin_var in integration_clin_vars) {
  integ_plots <- map(head(top_features, 3),
                     ~plot_metab_vs_clinical(.x, clin_var, analysis)) %>%
    Filter(Negate(is.null), .)
  if (length(integ_plots) > 0) {
    grid.arrange(grobs = integ_plots, ncol = min(3, length(integ_plots)), nrow = 1,
                 top = textGrob(paste0("Integración: Top metabólicas × ", clin_var),
                                gp = gpar(fontsize = 21, fontface = "bold", col = "#009E73")))
  }
}

# ── Heatmap integrado: z-score de mediana por subtipo × grupo ────────────────
if ("Subtype" %in% colnames(analysis) && length(top_features) > 0) {
  subtype_metab <- analysis %>%
    filter(!is.na(Subtype)) %>%
    select(Group, Subtype, all_of(intersect(top_features, colnames(analysis)))) %>%
    pivot_longer(cols = -c(Group, Subtype), names_to = "Feature", values_to = "Value") %>%
    group_by(Group, Subtype, Feature) %>%
    summarise(Median = median(Value, na.rm = TRUE), .groups = "drop") %>%
    group_by(Feature) %>%
    mutate(Median_z = scale(Median)[,1]) %>%
    ungroup() %>%
    mutate(GroupSubtype = paste0(Group, "\n", Subtype))

  if (nrow(subtype_metab) > 0) {
    print(
      ggplot(subtype_metab, aes(x = GroupSubtype, y = Feature, fill = Median_z)) +
        geom_tile(color = "white", linewidth = 0.6) +
        scale_fill_gradient2(
          low      = COL_CORE,
          mid      = "white",
          high     = COL_DIV,
          midpoint = 0,
          name     = "Mediana (z-score)"
        ) +
        labs(
          title    = "Firma Metabólica por Subtipo Molecular",
          subtitle = "Inversión metabólica sistemática entre grupos Core y Divergent",
          x        = "Grupo · Subtipo",
          y        = NULL
        ) +
        base_theme +
        theme(
          plot.title        = element_text(size = 24, face = "bold", hjust = 0.5),
          plot.subtitle     = element_text(size = 16, color = "grey40", hjust = 0.5),
          axis.text.x       = element_text(angle = 40, hjust = 1, size = 15),
          axis.text.y       = element_text(size = 14),
          axis.title.x      = element_text(size = 16, face = "bold"),
          legend.title      = element_text(size = 14, face = "bold"),
          legend.text       = element_text(size = 13),
          legend.key.width  = unit(1.5, "cm"),
          legend.key.height = unit(0.5, "cm"),
          legend.direction  = "horizontal",
          legend.position   = "bottom",
          panel.spacing     = unit(0.3, "lines")
        )
    )
  }
}

# ─── SECCIÓN IV: KAPLAN-MEIER ────────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN IV\nAnálisis de Supervivencia",
          gp = gpar(fontsize = 33, fontface = "bold", col = "#CC79A7"), y = 0.60)
grid.text("¿El fenotipo metabólico Core/Divergent predice pronóstico?",
          gp = gpar(fontsize = 19, col = "grey40"), y = 0.42)

if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>% filter(!is.na(OS.time), !is.na(OS), OS.time > 0)
  cat("\n📊 Kaplan-Meier: pacientes con datos de supervivencia:", nrow(km_data),
      "(Core:", sum(km_data$Group == "Core"),
      "| Divergent:", sum(km_data$Group == "Divergent"), ")\n")

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval = TRUE, pval.size = 6, conf.int = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE, risk.table.cex = 1.2,
                     surv.line.size = 1.2,
                     title          = "Curva de Supervivencia: Core vs Divergent",
                     xlab           = "Tiempo (días)",
                     ylab           = "Probabilidad de Supervivencia",
                     legend.labs    = c("Core", "Divergent"),
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 27, face = "bold"),
                             axis.title  = element_text(size = 21),
                             axis.text   = element_text(size = 19),
                             legend.text = element_text(size = 19))))

    # ── KM estratificado por Subtype ────────────────────────────────────────
    if ("Subtype" %in% colnames(km_data)) {
      subtypes_available <- km_data %>%
        filter(!is.na(Subtype)) %>%
        group_by(Subtype) %>%
        filter(n_distinct(Group) == 2, n() >= 10) %>%
        pull(Subtype) %>% unique()

      for (stype in subtypes_available) {
        km_sub <- km_data %>% filter(Subtype == stype)
        if (nrow(km_sub) < 10) next
        fit_sub <- survfit(Surv(OS.time, OS) ~ Group, data = km_sub)
        tryCatch(
          print(ggsurvplot(fit_sub, data = km_sub,
                           pval = TRUE, pval.size = 5, conf.int = TRUE,
                           palette   = unname(group_colors),
                           risk.table = TRUE, risk.table.cex = 1.1,
                           title     = paste0("Supervivencia: Core vs Divergent — Subtipo ", stype),
                           xlab      = "Tiempo (días)",
                           ylab      = "Probabilidad de Supervivencia",
                           legend.labs = c("Core", "Divergent"),
                           theme     = base_theme +
                             theme(plot.title = element_text(size = 22, face = "bold")))),
          error = function(e) warning(paste("KM subtipo", stype, "falló:", e$message))
        )
      }
    }

  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier.")
  }
} else {
  warning("⚠️  Columnas OS y/o OS.time no encontradas en analysis.")
}

dev.off()

cat("\n✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'\n")
cat("✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'\n")
cat("✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'\n")


🔍 Subsistemas SA detectados en CSV: 201 
   Subsistemas únicos: 85 
   • SA_Alanine_and_aspartate_metabolism
   • SA_Aminosugar_metabolism
   • SA_Androgen_and_estrogen_synthesis_and_metabolism
   • SA_Arachidonic_acid_metabolism
   • SA_Arginine_and_proline_metabolism
   • SA_Beta_Alanine_metabolism
   • SA_Bile_acid_synthesis
   • SA_Biomass_reactions
   • SA_Biotin_metabolism
   • SA_Butanoate_metabolism 
   ... 75 más

📊 Variables metabólicas encontradas: 248 / 286
⚠️  Variables NO encontradas ( 38 ):
   • LipidSat_pFBA
   • LipidUnsat_pFBA
   • CU_L2
   • EA_L2
   • WarburgIndex_L2
   • ATPConsumption_L2
   • ATPProduction_L2
   • RedoxIndex_L2
   • MFI_L2
   • AnabolismScore_L2
   • NADPHdemand_L2
   • TCA_completeness_L2
   • LipidSat_L2
   • LipidUnsat_L2
   • LipidPL_L2
   • GlnDependence_L2
   • CU_L2w
   • EA_L2w
   • WarburgIndex_L2w
   • ATPConsumption_L2w 
   ... 18 más

📋 Verificación de variables clave en analysis:
   ✅ OS.time                        → 1145/1166 con da

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Tamaño de efecto: Cliff's δ  |  ≥0.147 pequeño · ≥0.330 mediano · ≥0.474 grande' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Tamaño de efecto: Cliff's δ  |  ≥0.147 pequeño · ≥0.330 mediano · ≥0.474 g


📊 Kaplan-Meier: pacientes con datos de supervivencia: 1145 (Core: 1108 | Divergent: 37 )


Ignoring unknown labels:
• colour : "Strata"
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Core vs Divergent — Subtipo LumB' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Core vs Divergent — Subtipo LumA' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Core vs Divergent — Subtipo Missing' in 'mbcsToSbcs': - substituted for — (U+2014)”
Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Supervivencia: Core vs Divergent — Subtipo Basal' in 'mbcsToSbcs': - substituted for — (U+2014)”


pdf 
  2


✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'
✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'
✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'


In [8]:
install.packages("pheatmap")

Installing package into ‘/Users/eduardoruiz/Library/R/arm64/4.4/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/x0/j78w4f8n0_z4l575_k16n2k40000gn/T//RtmpPiHpQZ/downloaded_packages
